In [1]:
!pip install -q sentence-transformers

In [2]:
%%writefile guided_embeddings.py

Writing guided_embeddings.py


In [4]:

import numpy as np
from sentence_transformers import SentenceTransformer

from task2_chunks import docs, make_chunks


# 1. Create sentence-aware chunks

chunks = make_chunks(docs, "sentence")

print("\nTotal chunks:", len(chunks))


# 2. Load embedding model

print("\nLoading embedding model...")

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded successfully!")


# 3. Get text from chunks

chunk_texts = []

for chunk in chunks:
    chunk_texts.append(chunk["text"])


# 4. Generate document embeddings

print("\nGenerating document embeddings...")

document_embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Document embeddings generated!")


# 5. Three paraphrased queries

queries = [
    "I was sick before the deadline. Can I submit my assignment later?",
    "I cannot log into my student account. How can I get access again?",
    "My printer is producing blank pages. What should I check?"
]


# 6. Search function

def search_embeddings(query, top_k=3):

    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate cosine similarity

    scores = np.dot(
        document_embeddings,
        query_embedding
    )

    # Sort results by similarity score

    ranked_indexes = np.argsort(scores)[::-1]

    print("\nQuery:", query)
    print("-" * 60)

    for rank, index in enumerate(ranked_indexes[:top_k], start=1):

        chunk = chunks[index]

        print("\nResult:", rank)
        print("Chunk ID:", chunk["id"])
        print("Source:", chunk["source"])
        print("Similarity Score:", round(float(scores[index]), 4))

        print("Original Evidence:")
        print(chunk["text"])


# 7. Run all queries

print("\nEMBEDDING SEARCH RESULTS:")

for query in queries:
    search_embeddings(query)


# 8. Final message

print("\nAll embedding searches completed successfully.")



Total chunks: 21

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!

Generating document embeddings...
Document embeddings generated!

EMBEDDING SEARCH RESULTS:

Query: I was sick before the deadline. Can I submit my assignment later?
------------------------------------------------------------

Result: 1
Chunk ID: handbook:1
Source: handbook.txt
Similarity Score: 0.6111
Original Evidence:
Late submissions are normally penalized, but students who become ill before a deadline may request an extension if they contact the coordinator before the deadline and provide medical evidence. 

Result: 2
Chunk ID: handbook:3
Source: handbook.txt
Similarity Score: 0.4866
Original Evidence:
If the portal is unavailable, students should take a screenshot of the error and email the course coordinator with the assignment attached before the deadline. The coordinator can confirm whether the submission time will be adjusted. 

Result: 3
Chunk ID: handbook:0
Source: handbook.txt
Similarity Score: 0.484
Original Evidence:
University assignments sh